# Study 939 — DRIP or Sweep — the teardown

The dividend reconstruction and its two audits, the terminal-wealth gap with a Newey-West *t* on the daily log-return difference, 63-day circular block bootstrap CIs, an era cut, a rate-regime cut, sweeps of every labelled assumption, and a power calculation on the synthetic control.

Every real number is frozen from `docs/results.md` (Fingerprint `138ed0f07aec`, as-of 2026-06-30). Live cells are synthetic only and are labelled as such.

**Design.** Two arms over one holding. Shares are bought once at *t*₀ with 10,000; thereafter the arms differ only in what happens to a distribution. Exactly one execution lag: a purchase decided at the close of *t* is filled at the close of *t*+1. Costs are one-way × the amount reinvested (never × NAV); neither arm ever shorts, so no borrow applies. Sharpes are excess-of-cash (minus BIL's total return) on **both** arms. The simulation runs on the **price-only** close; the **total-return** close is used only to build the distribution stream and to audit it.

> 💡 **In plain words:** same fund, same shares, one difference — do you buy the moment the dividend lands, or let it sit in T-bills until quarter end?

In [1]:
R = {'asof': '2026-06-30', 'fp': '138ed0f07aec', 'start': '2007-05-30', 'end': '2026-06-30', 'n_spy': 4801, 'n_schd': 3692, 'pay_lag': 30, 'drip_cost': 0.0, 'sweep_cost': 2.0, 'rec': {'SPY': (77, 77, 77, 88.667, 88.854, 0.9979), 'VYM': (77, 77, 77, 45.2, 45.186, 1.0003), 'SCHD': (59, 59, 59, 8.97, 8.985, 0.9984)}, 'tr_audit': {'SPY': (0.99972, 0.042, -0.15), 'VYM': (0.99932, 0.068, -0.36), 'SCHD': (0.99889, 0.111, -0.75)}, 'spy': {'years': 19.1, 'sh_drip': 0.542, 'sh_sweep': 0.5424, 'w_drip': 69003, 'w_sweep': 68729, 'gap': 2.09, 't': 1.09, 'ci': (-1.18, 4.99), 'yld': 1.86}, 'vym': {'years': 19.1, 'sh_drip': 0.4865, 'sh_sweep': 0.4867, 'w_drip': 51370, 'w_sweep': 51033, 'gap': 3.46, 't': 1.18, 'ci': (-1.75, 7.97), 'yld': 3.12}, 'schd': {'years': 14.7, 'sh_drip': 0.7785, 'sh_sweep': 0.7801, 'w_drip': 60195, 'w_sweep': 59863, 'gap': 3.78, 't': 1.41, 'ci': (-1.74, 8.3), 'yld': 3.2}, 'matched': {'SPY': (1.77, 3.01, 1.87, 1.7), 'VYM': (3.09, 3.73, 1.4, 1.21), 'SCHD': (3.2, 3.78, 1.41, 1.18)}, 'hac': {'SPY': (1.09, 1.33, 1.57), 'VYM': (1.18, 1.42, 1.6), 'SCHD': (1.41, 1.46, 1.57)}, 'pooled_q': {'gap': 2.81, 't': 1.15, 'ci': (-1.47, 6.51), 'neg': 0.084}, 'pooled_a': {'gap': 12.78, 't': 1.94, 'ci': (0.04, 23.41), 'neg': 0.025}, 'annual': {'SPY': (9.18, 1.71), 'VYM': (15.46, 2.02), 'SCHD': (17.77, 2.73)}, 'era': {'SPY': ((1.68, 0.48), (1.84, 0.91)), 'VYM': ((3.55, 0.71), (2.43, 0.71)), 'SCHD': ((3.42, 1.09), (3.18, 0.91))}, 'rate': {'SPY': ((3.65, 1.28, 15), (6.05, 2.32, 4)), 'VYM': ((5.09, 1.28, 15), (8.49, 1.91, 4)), 'SCHD': ((5.61, 1.56, 11), (8.53, 1.34, 4))}, 'lag': {'SPY': (0.72, 3.63, 2.09, 1.23), 'VYM': (0.89, 4.11, 3.46, 1.59), 'SCHD': (0.8, 6.96, 3.78, 2.47)}, 'cost': {'SPY': (2.05, 2.09, 2.14, 2.23, 2.05), 'VYM': (3.4, 3.46, 3.54, 3.69, 3.39), 'SCHD': (3.72, 3.78, 3.87, 4.01, 3.71)}, 'money': {'SPY': (274, 14.4), 'VYM': (337, 17.7), 'SCHD': (332, 22.7)}, 'syn_q_planted': (14.23, 1.6, 0.56), 'syn_q_null': (0.03, 1.71, 0.6), 'syn_a_planted': (52.13, 5.28, 1.87), 'syn_a_null': (-1.03, 6.12, 2.16), 'syn_mono': ((3, 8.8), (6, 13.72), (9, 20.88)), 'power_planted': (1.5, 2.26, 0.8), 'power_null': (-0.52, 2.3, 0.81)}

## 1. The reconstruction

`D_t = P_{t−1}·(TR_t / TR_{t−1}) − P_t`, thresholded at 5 bp of price to reject the float dust that survives the two legs' independent rounding. The vendor's reported cash column is used **only** to score this — never as an input.

> 💡 **In plain words:** the adjusted-close series already contains the dividends. Divide it by the unadjusted one and the payments fall out.

In [2]:
for tk in ('SPY','VYM','SCHD'):
    n_rec, n_rep, n_match, tot_rec, tot_rep, ratio = R['rec'][tk]
    tr_ratio, dev, track = R['tr_audit'][tk]
    print(f"{tk:5s} events {n_rec:3d} rec / {n_rep:3d} reported / {n_match:3d} matched  |  "
          f"cash/share {tot_rec:8.3f} vs {tot_rep:8.3f} (ratio {ratio:.4f})")
    print(f"      zero-lag DRIP vs the TR index: terminal ratio {tr_ratio:.5f}, "
          f"max dev {dev:.3f}%, tracking {track:+.2f} bps/yr")

SPY   events  77 rec /  77 reported /  77 matched  |  cash/share   88.667 vs   88.854 (ratio 0.9979)
      zero-lag DRIP vs the TR index: terminal ratio 0.99972, max dev 0.042%, tracking -0.15 bps/yr
VYM   events  77 rec /  77 reported /  77 matched  |  cash/share   45.200 vs   45.186 (ratio 1.0003)
      zero-lag DRIP vs the TR index: terminal ratio 0.99932, max dev 0.068%, tracking -0.36 bps/yr
SCHD  events  59 rec /  59 reported /  59 matched  |  cash/share    8.970 vs    8.985 (ratio 0.9984)
      zero-lag DRIP vs the TR index: terminal ratio 0.99889, max dev 0.111%, tracking -0.75 bps/yr


The second line is the study's load-bearing audit: a **zero-lag, zero-cost DRIP is the definition of the adjusted close**, so it has to reproduce it. It does, to within 0.04-0.11% over 15-19 years; the residual is the single execution lag.

## 2. The headline race — DRIP vs quarterly sweep

Excess-of-cash Sharpes on both arms, terminal wealth from 10,000, 63-day block bootstrap (2,000 draws) on the annualised log-wealth gap.

In [3]:
for tag, d in [('SPY', R['spy']), ('VYM', R['vym']), ('SCHD', R['schd'])]:
    lo, hi = d['ci']
    print(f"{tag:5s} {d['years']:.1f}yr  exSharpe DRIP {d['sh_drip']:+.4f} / "
          f"SWEEP {d['sh_sweep']:+.4f}   W {d['w_drip']:,} vs {d['w_sweep']:,}")
    print(f"      gap {d['gap']:+.2f} bps/yr  HAC t {d['t']:+.2f}  "
          f"95% CI [{lo:+.2f}, {hi:+.2f}]")
p = R['pooled_q']
print(f"\npooled (equal-weight): {p['gap']:+.2f} bps/yr  HAC t {p['t']:+.2f}  "
      f"CI [{p['ci'][0]:+.2f}, {p['ci'][1]:+.2f}]  share of resamples < 0: {p['neg']:.1%}")
print('note: the excess-of-cash SHARPE race is a dead heat and runs the other way')
print('      in the 4th decimal - the swept arm carries a sliver of cash ballast.')

SPY   19.1yr  exSharpe DRIP +0.5420 / SWEEP +0.5424   W 69,003 vs 68,729
      gap +2.09 bps/yr  HAC t +1.09  95% CI [-1.18, +4.99]
VYM   19.1yr  exSharpe DRIP +0.4865 / SWEEP +0.4867   W 51,370 vs 51,033
      gap +3.46 bps/yr  HAC t +1.18  95% CI [-1.75, +7.97]
SCHD  14.7yr  exSharpe DRIP +0.7785 / SWEEP +0.7801   W 60,195 vs 59,863
      gap +3.78 bps/yr  HAC t +1.41  95% CI [-1.74, +8.30]

pooled (equal-weight): +2.81 bps/yr  HAC t +1.15  CI [-1.47, +6.51]  share of resamples < 0: 8.4%
note: the excess-of-cash SHARPE race is a dead heat and runs the other way
      in the 4th decimal - the swept arm carries a sliver of cash ballast.


Two things the headline table does **not** get to claim.

First, the *t*-statistics are not an artefact of a short HAC kernel — the automatic lag (≈9 days) gives the *lowest* reading of the three tested, and even a 252-day kernel leaves every fund under 1.6.

Second, the ordering by distribution yield is far weaker than it looks. SPY and VYM are raced over 2007-2026 and SCHD only over 2011-2026, so the triple mixes *yield* with *window*. Re-race all three on SCHD's window and the spread collapses; normalised by realised yield, the ordering **reverses**.

In [4]:
print('HAC lag robustness of the headline t (auto is the LOWEST reading):')
print(f"{'':7s}{'auto':>8s}{'63d':>8s}{'252d':>8s}")
for tk in ('SPY','VYM','SCHD'):
    print('  %-5s' % tk + ''.join('%8.2f' % x for x in R['hac'][tk]))
print('\nsame race on the MATCHED window (SCHD start -> 2026-06-30):')
print(f"{'':7s}{'yield':>8s}{'gap':>8s}{'t':>8s}{'gap/1% yld':>12s}")
for tk in ('SPY','VYM','SCHD'):
    y, g, t, per = R['matched'][tk]
    print(f"  {tk:<5s}{y:7.2f}%{g:8.2f}{t:8.2f}{per:12.2f}")
print('-> the ordering survives, the spread does not, and per unit of yield')
print('   the LOW-yield fund shows the LARGEST gap. Most of the headline')
print('   spread is SPY carrying an extra ZIRP half, not yield.')

HAC lag robustness of the headline t (auto is the LOWEST reading):
           auto     63d    252d
  SPY      1.09    1.33    1.57
  VYM      1.18    1.42    1.60
  SCHD     1.41    1.46    1.57

same race on the MATCHED window (SCHD start -> 2026-06-30):
          yield     gap       t  gap/1% yld
  SPY     1.77%    3.01    1.87        1.70
  VYM     3.09%    3.73    1.40        1.21
  SCHD    3.20%    3.78    1.41        1.18
-> the ordering survives, the spread does not, and per unit of yield
   the LOW-yield fund shows the LARGEST gap. Most of the headline
   spread is SPY carrying an extra ZIRP half, not yield.


## 3. Sweep frequency — the only cut with a pulse

In [5]:
for tk in ('SPY','VYM','SCHD'):
    q = R['spy' if tk=='SPY' else ('vym' if tk=='VYM' else 'schd')]
    ga, ta = R['annual'][tk]
    print(f"{tk:5s} quarterly {q['gap']:+6.2f} bps/yr (t {q['t']:+.2f})   "
          f"annual {ga:+6.2f} bps/yr (t {ta:+.2f})")
pa = R['pooled_a']
print(f"pooled annual: {pa['gap']:+.2f} bps/yr (t {pa['t']:+.2f}), "
      f"CI [{pa['ci'][0]:+.2f}, {pa['ci'][1]:+.2f}] - lower edge sits ON zero")

SPY   quarterly  +2.09 bps/yr (t +1.09)   annual  +9.18 bps/yr (t +1.71)
VYM   quarterly  +3.46 bps/yr (t +1.18)   annual +15.46 bps/yr (t +2.02)
SCHD  quarterly  +3.78 bps/yr (t +1.41)   annual +17.77 bps/yr (t +2.73)
pooled annual: +12.78 bps/yr (t +1.94), CI [+0.04, +23.41] - lower edge sits ON zero


> 💡 **In plain words:** waiting until quarter end costs nothing, because the pay-date lag has already used up most of the delay. Waiting a whole year costs about as much as a fund fee.

## 4. Era and rate-regime cuts

In [6]:
print('era cut (split 2016-01-01), bps/yr. SPY/VYM start 2007 (BIL inception),')
print('SCHD starts 2011, so its early half is only 4.2 years:')
for tk in ('SPY','VYM','SCHD'):
    (g1,t1),(g2,t2) = R['era'][tk]
    print(f"  {tk:5s} early {g1:+.2f} (t {t1:+.2f})   late(2016-2026) {g2:+.2f} (t {t2:+.2f})")
print('\nrate-regime cut (calendar years by realised BIL return, 2% threshold):')
for tk in ('SPY','VYM','SCHD'):
    (gl,tl,nl),(gh,th,nh) = R['rate'][tk]
    print(f"  {tk:5s} low-rate  {gl:+.2f} (t {tl:+.2f}, {nl} yr)   "
          f"high-rate {gh:+.2f} (t {th:+.2f}, {nh} yr)")

era cut (split 2016-01-01), bps/yr. SPY/VYM start 2007 (BIL inception),
SCHD starts 2011, so its early half is only 4.2 years:
  SPY   early +1.68 (t +0.48)   late(2016-2026) +1.84 (t +0.91)
  VYM   early +3.55 (t +0.71)   late(2016-2026) +2.43 (t +0.71)
  SCHD  early +3.42 (t +1.09)   late(2016-2026) +3.18 (t +0.91)

rate-regime cut (calendar years by realised BIL return, 2% threshold):
  SPY   low-rate  +3.65 (t +1.28, 15 yr)   high-rate +6.05 (t +2.32, 4 yr)
  VYM   low-rate  +5.09 (t +1.28, 15 yr)   high-rate +8.49 (t +1.91, 4 yr)
  SCHD  low-rate  +5.61 (t +1.56, 11 yr)   high-rate +8.53 (t +1.34, 4 yr)


Positive in **both** halves on all three funds: the sign is era-robust even though the magnitude never becomes significant.

The rate-regime cut refutes the naive story. "DRIP wins most when cash pays nothing" is **backwards on this tape** — the high-rate bucket (2023-2026) shows the *larger* gap, because those were also enormous equity years. The DRIP advantage is the realised **equity-minus-cash spread** over the delay; the level of the bill rate says nothing about that spread on its own. (Caveat: the buckets are unions of non-contiguous calendar years, so the wealth paths inside each are spliced — read the sign, not the decimals.)

## 5. The assumption sweeps

Three inputs are **not on the tape** and are labelled as assumptions: the pay lag, the DRIP cost and the sweep cost. Tax is ignored deliberately: both arms receive the same distribution, of the same size, on the same date, so the bulk of the tax bill is common. What is *not* common — the sweep arm's bill interest, taxed as ordinary income, and the two arms' different capital-gain lots — is second-order here and runs *against* the sweep arm, so ignoring tax understates DRIP's edge rather than manufacturing it.

In [7]:
print('pay-lag sweep (ASSUMPTION: Yahoo publishes ex-dates, not pay dates), bps/yr:')
print(f"{'':7s}{'0d':>8s}{'15d':>8s}{'30d*':>8s}{'45d':>8s}   (* = headline)")
for tk in ('SPY','VYM','SCHD'):
    print('  %-5s' % tk + ''.join('%8.2f' % x for x in R['lag'][tk]))
print('\ncost sweep (drip_bps, sweep_bps), one-way x amount reinvested, bps/yr:')
print(f"{'':7s}{'(0,0)':>9s}{'(0,2)*':>9s}{'(0,5)':>9s}{'(0,10)':>9s}{'(5,5)':>9s}")
for tk in ('SPY','VYM','SCHD'):
    print('  %-5s' % tk + ''.join('%9.2f' % x for x in R['cost'][tk]))

pay-lag sweep (ASSUMPTION: Yahoo publishes ex-dates, not pay dates), bps/yr:
             0d     15d    30d*     45d   (* = headline)
  SPY      0.72    3.63    2.09    1.23
  VYM      0.89    4.11    3.46    1.59
  SCHD     0.80    6.96    3.78    2.47

cost sweep (drip_bps, sweep_bps), one-way x amount reinvested, bps/yr:
           (0,0)   (0,2)*    (0,5)   (0,10)    (5,5)
  SPY       2.05     2.09     2.14     2.23     2.05
  VYM       3.40     3.46     3.54     3.69     3.39
  SCHD      3.72     3.78     3.87     4.01     3.71


**This is the finding that sets the Tradability stamp.** A 0-10 bp cost sweep moves the gap by under 0.2 bps; the *unobservable* pay lag moves it from +0.7 to +7.0 bps/yr. When the assumption you had to guess has a wider range than the quantity you measured, you do not have a measurement.

## 6. Live synthetic control — the machinery is unbiased

**Synthetic tape below, not the real tape.** The generator's defaults are a deliberately loud lab bench (20% premium over cash, 6% distribution yield, a quiet 6% vol, 20 years) chosen so that an effect worth a few bps a year is resolvable at all. Planted: DRIP must win. Null (the fund drifts at exactly the cash rate): the gap must vanish.

In [8]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from drip_sweep import data, strategy as st
for freq in ('Q','A'):
    pl = st.seed_sweep(data.synthetic_daily, 1.0, n_seeds=8, sweep_freq=freq)
    nl = st.seed_sweep(data.synthetic_daily, 0.0, n_seeds=8, sweep_freq=freq)
    print(f"{freq}: planted {pl['mean']:+7.2f} bps/yr (se {pl['se']:.2f})   "
          f"null {nl['mean']:+7.2f} bps/yr (se {nl['se']:.2f})")
frames, truth = data.synthetic_panel(signal_strength=1.0, seed=939)
print('\nmonotone in planted distribution yield (quarterly sweep):')
for tk, y in zip(truth['tickers'], truth['div_yields']):
    g = st.synthetic_detect(frames[tk])['gap_bps_per_year']
    print(f"  yield {y:.0%}: gap {g:+.2f} bps/yr")

Q: planted  +14.23 bps/yr (se 0.56)   null   +0.03 bps/yr (se 0.60)


A: planted  +52.13 bps/yr (se 1.87)   null   -1.03 bps/yr (se 2.16)

monotone in planted distribution yield (quarterly sweep):


  yield 3%: gap +8.80 bps/yr
  yield 6%: gap +13.72 bps/yr


  yield 9%: gap +20.88 bps/yr


## 7. The power calculation that settles the stamp

**Synthetic tape below.** Same detector, market-realistic parameters: a 5.5% premium over cash, a 3% distribution yield, 16% volatility, twenty years. If the detector cannot separate a *true* effect of that size from zero on one path, then a real-tape *t* of 1.2 is not evidence of absence — it is the expected reading.

In [9]:
real = dict(equity_premium=0.055, div_yield_ann=0.03, vol_ann=0.16)
pl = st.seed_sweep(data.synthetic_daily, 1.0, n_seeds=8, gen_kw=real)
nl = st.seed_sweep(data.synthetic_daily, 0.0, n_seeds=8, gen_kw=real)
sep = pl['mean'] - nl['mean']
pooled_se = np.sqrt(pl['se']**2 + nl['se']**2)
print(f"planted {pl['mean']:+.2f} bps/yr (sd {pl['sd']:.2f}, se {pl['se']:.2f})")
print(f"null    {nl['mean']:+.2f} bps/yr (sd {nl['sd']:.2f}, se {nl['se']:.2f})")
print(f"separation {sep:+.2f} bps/yr against a per-tape sd of ~{pl['sd']:.2f}")
print(f"-> a SINGLE 20-year tape has |t| ~ {abs(sep)/pl['sd']:.2f} against the null.")
print('   The real tape reads +1.09 to +1.41. That is exactly this.')

planted +1.50 bps/yr (sd 2.26, se 0.80)
null    -0.52 bps/yr (sd 2.30, se 0.81)
separation +2.03 bps/yr against a per-tape sd of ~2.26
-> a SINGLE 20-year tape has |t| ~ 0.90 against the null.
   The real tape reads +1.09 to +1.41. That is exactly this.


## Verdict

- **Signal — Weak.** The DRIP-minus-sweep gap is **+2.09 / +3.46 / +3.78 bps/yr** (SPY / VYM / SCHD), pooled **+2.81**, with HAC *t* of +1.09 / +1.18 / +1.41 (pooled +1.15) and bootstrap CIs that all include zero (and the auto-lag *t* is the lowest of the HAC kernels tested, so nothing is being hidden by a short window). The effect is correctly signed, positive in both eras, ordered by distribution yield — though that ordering largely evaporates on matched windows and *reverses* per unit of yield — and the synthetic control recovers a planted version at *t* ≈ 25 while staying silent on the null (+0.03 ± 0.60 bps/yr) — but the desk's bar is a robust |*t*| ≥ 2 on the **real tape**, and only the annual-sweep variant reaches it (SCHD +17.77 bps/yr, *t* = +2.73). The power check shows why: at realistic parameters one twenty-year path cannot resolve this. **Survivorship** is named — three large surviving US ETFs, picked because they are what people own.
- **Tradability — Mirage.** +2.81 bps/yr is 14-23 currency units a year on 10,000 — below one ETF ticket's half-spread, invariant to a 0-10 bp cost sweep, and swamped by the pay-lag assumption's own +0.7 to +7.0 bps/yr range. Nothing here is bankable. The defensible advice is administrative: **DRIP is free and removes four decisions a year**; if you must sweep, sweep **quarterly** (≈3 bps/yr) rather than **annually** (≈13 bps/yr).